In [4]:
# Install necessary packages
!pip install pandas plotly folium geopandas shapely

# Step 1: Upload multiple CSVs
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

# Combine all uploaded CSVs into a single DataFrame
dfs = []
for filename in uploaded.keys():
    dfs.append(pd.read_csv(io.BytesIO(uploaded[filename])))

migrant_df = pd.concat(dfs, ignore_index=True)

# Preview the data
migrant_df.head()


Saving final_df_with_climate_and_policy.csv to final_df_with_climate_and_policy.csv
Saving migrant_incidents.csv to migrant_incidents.csv


,Incident Date,year,month,state,Total Number of Dead and Missing,Cause of Death,average_temp,Migration Route,geometry,Main ID,...,Number of Children,Country of Origin,Region of Origin,Country of Incident,Location of Incident,Coordinates,UNSD Geographical Grouping,Information Source,URL,Source Quality
0,1/6/2014,2014.0,1.0,ARIZONA,1,Mixed or unknown,44.0,US-Mexico border crossing,POINT (-110.366453 31.650259),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1/12/2014,2014.0,1.0,ARIZONA,1,Mixed or unknown,44.0,US-Mexico border crossing,POINT (-111.73756 31.59713),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1/14/2014,2014.0,1.0,NaN,1,Mixed or unknown,NaN,US-Mexico border crossing,POINT (-113.01125 31.94026),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1/16/2014,2014.0,1.0,ARIZONA,1,Violence,44.0,US-Mexico border crossing,POINT (-109.315632 31.506777),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1/17/2014,2014.0,1.0,ARIZONA,1,Mixed or unknown,44.0,US-Mexico border crossing,POINT (-113.18402 32.45435),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Upload files
uploaded = files.upload()

# Read CSVs
dfs = [pd.read_csv(io.BytesIO(uploaded[k])) for k in uploaded.keys()]
migrant_df = pd.concat(dfs, ignore_index=True)

# Check uploaded columns
print("Uploaded Columns:", migrant_df.columns.tolist())

# Correct Date parsing
migrant_df['Date'] = pd.to_datetime(migrant_df['Incident Date'], errors='coerce')

# Now proceed with all the dashboard and visualizations


Saving final_df_with_climate_and_policy.csv to final_df_with_climate_and_policy (1).csv
Saving migrant_incidents.csv to migrant_incidents (1).csv
Uploaded Columns: ['Incident Date', 'year', 'month', 'state', 'Total Number of Dead and Missing', 'Cause of Death', 'average_temp', 'Migration Route', 'geometry', 'Main ID', 'Incident ID', 'Incident Type', 'Region of Incident', 'Incident Year', 'Month', 'Number of Dead', 'Minimum Estimated Number of Missing', 'Number of Survivors', 'Number of Females', 'Number of Males', 'Number of Children', 'Country of Origin', 'Region of Origin', 'Country of Incident', 'Location of Incident', 'Coordinates', 'UNSD Geographical Grouping', 'Information Source', 'URL', 'Source Quality']


In [6]:
# Ensure Date is parsed correctly
migrant_df['Date'] = pd.to_datetime(migrant_df['Date'], errors='coerce')

# Create a 'Season' column
def get_season(date):
    if pd.isnull(date):
        return 'Unknown'
    month = date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

migrant_df['Season'] = migrant_df['Date'].apply(get_season)

# Confirm it looks good
migrant_df[['Date', 'Season']].head()


,Date,Season
0,2014-01-06,Winter
1,2014-01-12,Winter
2,2014-01-14,Winter
3,2014-01-16,Winter
4,2014-01-17,Winter


In [7]:
# Install necessary packages
!pip install plotly

# Imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Step 1: Load Data
# (assuming file already uploaded to migrant_df)
# migrant_df = pd.read_csv('/content/migrant_incidents.csv')

# Step 2: Date parsing
migrant_df['Date'] = pd.to_datetime(migrant_df['Incident Date'], errors='coerce')

# Clean missing numeric values
migrant_df['Total Number of Dead and Missing'] = migrant_df['Total Number of Dead and Missing'].fillna(0)
migrant_df['average_temp'] = migrant_df['average_temp'].fillna(0)

# Step 3: Derive Year and Season
migrant_df['Year'] = migrant_df['Date'].dt.year
migrant_df['Season'] = migrant_df['Date'].dt.month % 12 // 3 + 1
season_dict = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'}
migrant_df['Season'] = migrant_df['Season'].map(season_dict)

# Step 4: Group for Seasonal Chart
seasonal_counts = migrant_df.groupby(['Year', 'Season']).agg({
    'Total Number of Dead and Missing': 'sum'
}).reset_index()

seasonal_counts.rename(columns={'Total Number of Dead and Missing': 'Disappearance_Count'}, inplace=True)

# --- PART 1: SEASONAL VARIATION LINE CHART ---

fig1 = px.line(
    seasonal_counts,
    x='Year',
    y='Disappearance_Count',
    color='Season',
    markers=True,
    title='Seasonal Trend of Migrant Disappearances',
    labels={'Disappearance_Count': 'Number of Disappearances'},
    hover_data={'Year': True, 'Season': True, 'Disappearance_Count': True}
)

fig1.update_layout(
    font=dict(size=14),
    title_font_size=22,
    hovermode="x unified",
    legend_title_text='Season',
    plot_bgcolor='rgb(245,245,245)',
    paper_bgcolor='white'
)

fig1.show()

# --- PART 2: DUAL Y-AXIS PLOT (DISAPPEARANCES vs AVERAGE TEMP) ---

# Group for Dual Axis
region_grouped = migrant_df.groupby(['Year', 'Region of Incident']).agg({
    'Total Number of Dead and Missing': 'sum',
    'average_temp': 'mean'
}).reset_index()

month_grouped = migrant_df.groupby(['Year', 'month']).agg({
    'Total Number of Dead and Missing': 'sum',
    'average_temp': 'mean'
}).reset_index()

# Prepare All Data (for 'All Regions/All Months')
all_yearly = migrant_df.groupby('Year').agg({
    'Total Number of Dead and Missing': 'sum',
    'average_temp': 'mean'
}).reset_index()

all_yearly.rename(columns={'Total Number of Dead and Missing': 'Disappearance_Count',
                            'average_temp': 'Average_Temperature'}, inplace=True)

# --- Create Base Figure ---
fig2 = go.Figure()

# Disappearance Trace
fig2.add_trace(go.Scatter(
    x=all_yearly['Year'],
    y=all_yearly['Disappearance_Count'],
    name="Disappearances",
    line=dict(color='firebrick', width=3),
    yaxis="y1",
    mode='lines+markers'
))

# Temperature Trace
fig2.add_trace(go.Scatter(
    x=all_yearly['Year'],
    y=all_yearly['Average_Temperature'],
    name="Average Temperature (°C)",
    line=dict(color='royalblue', width=3, dash='dot'),
    yaxis="y2",
    mode='lines+markers'
))

# Highlight Extreme Climate Years
highlight_years = [2015, 2022]

for year in highlight_years:
    fig2.add_vrect(
        x0=year-0.5, x1=year+0.5,
        fillcolor="LightSalmon", opacity=0.3,
        layer="below", line_width=0,
        annotation_text=f"Extreme {year}",
        annotation_position="top left",
        annotation_font_size=12
    )

# --- PART 3: DROPDOWN FILTERS (REGION + MONTH) ---

# Available filters
available_regions = migrant_df['Region of Incident'].dropna().unique()
available_months = sorted(migrant_df['month'].dropna().unique())

# Build Dropdown Buttons
dropdown_buttons = []

# Region Filter Buttons
for region in available_regions:
    filtered = region_grouped[region_grouped['Region of Incident'] == region]

    disappearance_trace = go.Scatter(
        x=filtered['Year'],
        y=filtered['Total Number of Dead and Missing'],
        name="Disappearances",
        line=dict(color='firebrick', width=3),
        yaxis="y1",
        mode='lines+markers'
    )

    temp_trace = go.Scatter(
        x=filtered['Year'],
        y=filtered['average_temp'],
        name="Average Temperature (°C)",
        line=dict(color='royalblue', width=3, dash='dot'),
        yaxis="y2",
        mode='lines+markers'
    )

    dropdown_buttons.append(dict(
        label=f"Region: {region}",
        method='update',
        args=[{
            'x': [filtered['Year'], filtered['Year']],
            'y': [filtered['Total Number of Dead and Missing'], filtered['average_temp']],
        },
        {
            'title': f"Migrant Disappearances and Climate Factors - {region}"
        }]
    ))

# Month Filter Buttons
for month in available_months:
    filtered = month_grouped[month_grouped['month'] == month]

    dropdown_buttons.append(dict(
        label=f"Month: {int(month)}",
        method='update',
        args=[{
            'x': [filtered['Year'], filtered['Year']],
            'y': [filtered['Total Number of Dead and Missing'], filtered['average_temp']],
        },
        {
            'title': f"Migrant Disappearances and Climate Factors - Month {int(month)}"
        }]
    ))

# Add "All" button
dropdown_buttons.insert(0, dict(
    label="All (No Filter)",
    method='update',
    args=[{
        'x': [all_yearly['Year'], all_yearly['Year']],
        'y': [all_yearly['Disappearance_Count'], all_yearly['Average_Temperature']],
    },
    {
        'title': "Migrant Disappearances and Climate Factors (All Data)"
    }]
))

# Attach Dropdown
fig2.update_layout(
    updatemenus=[{
        'buttons': dropdown_buttons,
        'direction': 'down',
        'showactive': True,
        'x': 1.15,
        'xanchor': 'left',
        'y': 1.2,
        'yanchor': 'top'
    }]
)

# Final Layout
fig2.update_layout(
    title="Migrant Disappearances and Climate Factors (Dual Y-Axis, Highlighted Events)",
    xaxis=dict(
        title="Year",
        tickmode='linear',
        dtick=1
    ),
    yaxis=dict(
        title="Number of Disappearances",
        titlefont=dict(color="firebrick"),
        tickfont=dict(color="firebrick")
    ),
    yaxis2=dict(
        title="Average Temperature (°C)",
        titlefont=dict(color="royalblue"),
        tickfont=dict(color="royalblue"),
        overlaying="y",
        side="right"
    ),
    font=dict(
        size=14
    ),
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0)'
    ),
    plot_bgcolor='rgb(245,245,245)',
    paper_bgcolor='white',
    hovermode="x unified"
)

# Show
fig2.show()


In [10]:
# Import plotly.io
import plotly.io as pio

# --- Save fig1 (Seasonal Trend Line Chart) as HTML ---
pio.write_html(fig1, file="/content/seasonal_trend_migrant_disappearances.html", auto_open=False)

# --- Save fig2 (Dual Y-Axis + Dropdown) as HTML ---
pio.write_html(fig2, file="/content/dual_axis_migrant_disappearances_temperature.html", auto_open=False)

# --- Download the HTML files ---
from google.colab import files

# Download each file
files.download("/content/seasonal_trend_migrant_disappearances.html")
files.download("/content/dual_axis_migrant_disappearances_temperature.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


1. **Seasonal Trend Line Chart: Migrant Disappearances by Season**

This second visualization depicts a multi-line interactive line chart showing the seasonal trends in migrant disappearances over time. The X-axis represents the year, while the Y-axis captures the number of migrant disappearances recorded in that period. Each season—Winter, Spring, Summer, and Fall—is represented by a distinct color, enabling users to observe how disappearance patterns shift across different times of the year. The chart is fully interactive: hovering over any point reveals precise disappearance counts for that season and year, while clicking on the legend allows users to isolate or compare specific seasonal trends.

The significance of this visualization lies in its ability to surface seasonal migration vulnerabilities. It highlights, for instance, whether summer months see higher disappearance rates due to factors such as heat waves, dehydration, or increased sea crossings, or whether winter brings different sets of challenges. By visualizing these patterns, the chart empowers humanitarian actors and policymakers to time interventions more effectively, designing seasonally responsive migration support strategies.

Methodologically, the underlying data was processed by extracting the season from each incident's date, grouping the disappearances by year and season, and calculating the total number of missing or deceased migrants per combination. The chart was designed using Plotly Express (px.line), allowing for elegant hover interactivity, clear legends, and smooth background integration. Like the dual-axis plot, it was exported as a standalone interactive HTML file to ensure portability and ease of sharing.

2. **Dual Axis Plot: Migrant Disappearances vs Average Temperature**

This visualization presents a dual Y-axis interactive line chart that tracks two key variables over time: the number of migrant disappearances and the average temperature recorded per year. On the left Y-axis, the chart displays the annual total of migrant disappearances, while the right Y-axis tracks the corresponding average temperature (in degrees Celsius). Users can dynamically filter the data by selecting specific Regions of Incident or Months from the interactive dropdown menus, allowing for granular exploration of migration risks across different geographic and temporal scales. Certain critical years, such as 2015 and 2022, are highlighted with shaded vertical bands to mark periods of extreme climatic events, prompting viewers to consider how environmental shocks may intensify vulnerabilities along migration routes.

The significance of this visualization lies in its ability to reveal potential correlations between climatic stress and human displacement. By bringing together disappearance counts and temperature trends, it encourages viewers to think critically about the ways in which environmental degradation, rising heat levels, and seasonal extremes intersect with humanitarian crises. Methodologically, the data was grouped by year and region/month, with total migrant disappearances and average temperatures calculated for each interval. The use of vertical annotations ensures that key climate years are visually emphasized.

Technically, this visualization was created using Plotly Graph Objects (go.Figure), leveraging dual Y-axes, dynamic dropdown filters, and shaded rectangles for highlighting. It was exported as a fully interactive HTML file, making it accessible for offline analysis or integration into reports. The broader objective is to allow dynamic, user-driven exploration of whether, when, and where climate variability amplifies migrant vulnerabilities, supporting both academic research and policy discussions around climate justice and migration resilience.

The overarching objective of this visualization is to provide historical insights into how seasonal rhythms interact with migration risks. By tracing fluctuations in disappearances across winter, spring, summer, and fall, the chart offers a deeper understanding of the environmental dimensions of migration, shedding light on periods of intensified vulnerability and the necessity of seasonally attuned responses.



In [32]:
from IPython.core.display import HTML

# CSS styling
style = """
<style>
  body {
    background-color: #f7f7f7;
    font-family: 'Segoe UI', 'Roboto', 'Arial', sans-serif;
    color: #333333;
    margin: 20px;
  }

  h1, h2, h3 {
    color: #222222;
    text-align: center;
    font-weight: 600;
  }

  .plotly-graph-div {
    margin: auto;
    max-width: 90%;
    box-shadow: 0 4px 8px rgba(0,0,0,0.1);
    border-radius: 8px;
    background-color: white;
    padding: 20px;
  }

  a {
    color: #0056b3;
    text-decoration: none;
  }

  a:hover {
    text-decoration: underline;
  }
</style>
"""

# Display the CSS styling
HTML(style)


In [37]:
# prompt: download QUESTION2.ipynb as html

from google.colab import files
files.download('QUESTION2 (2).ipynb')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
!jupyter nbconvert --to html 'QUESTION2 (2).ipynb''

/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [39]:
!download QUESTION2.html

/bin/bash: line 1: download: command not found


In [40]:
from google.colab import files
files.download('QUESTION2.html')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>